# 07 — MongoDB: Documentos e CRUD

**Objetivos:** modelar documentos e usar CRUD real opcional com PyMongo

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia técnico — MongoDB: fundamentos e CRUD](./reference_materials/07_mongodb_fundamentos_crud.pdf)
- [Cheat sheet — MongoDB: fundamentos e CRUD](./cheat_sheets/cheat_sheet_mongodb_fundamentos_crud.pdf)
- [Documentação e arquivos de resultados](./README.pt-BR.md)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## Documentos e estrutura

MongoDB armazena documentos BSON. Arrays e subdocumentos são úteis quando o padrão de acesso é coeso; referências são preferíveis para relações de alta cardinalidade e atualizações independentes.

In [2]:
import json
usuarios=json.loads((DATA/'usuarios_sinteticos.json').read_text(encoding='utf8'));display(pd.DataFrame(usuarios).head())

,_id,nome,cidade,interesses,ativo
0,1,Ana,Belo Horizonte,"[dados, livros]",True
1,2,Bruno,Rio de Janeiro,[esportes],True
2,3,Carla,São Paulo,"[dados, café]",False
3,4,Diego,Belo Horizonte,"[música, dados]",True
4,5,Eva,Curitiba,[livros],True


## Operações modernas

`insert_many`, `find`, `update_one` com `$set` e `delete_one` substituem APIs antigas. O exemplo real depende de MongoDB local e da variável `RUN_LIVE_DB=1`.

In [3]:
# A execução local não ativa conexões automaticamente.
# Para laboratórios opcionais, configure RUN_LIVE_DB=1 antes de iniciar o kernel,
# depois de verificar que a instância é EXCLUSIVAMENTE de estudos.
import os
print('Laboratório externo:', 'habilitado' if os.getenv('RUN_LIVE_DB') == '1' else 'desabilitado (padrão seguro)')

Laboratório externo: desabilitado (padrão seguro)


In [4]:
# IMPORTANTE: só execute em instância de laboratório isolada.
# Coleção única para esta sessão: nenhuma exclusão de dados anteriores.
import os
if os.getenv('RUN_LIVE_DB') == '1':
    from uuid import uuid4
    from pymongo import MongoClient
    with MongoClient('mongodb://127.0.0.1:27017',serverSelectionTimeoutMS=3000) as cli:
        cli.admin.command('ping')
        nome = 'aula07_' + uuid4().hex[:12]
        colecao = cli['ep04_laboratorio'][nome]
        colecao.insert_many(usuarios)
        colecao.update_one({'_id':1},{'$set':{'ativo':False}})
        print('Coleção criada nesta sessão:',nome)
        print(list(colecao.find({'interesses':'dados'},{'_id':0,'nome':1,'cidade':1})))
else:
    print('Laboratório MongoDB não executado (RUN_LIVE_DB diferente de 1).')

Laboratório MongoDB não executado (RUN_LIVE_DB diferente de 1).


## Exercício resolvido — filtrar documentos

Buscar usuários ativos de uma cidade e projetar apenas os nomes. Usar uma lista Python aqui apenas para conferir a lógica dos dados; **não é substituto da execução MongoDB**.

In [5]:
resultado=[u['nome'] for u in usuarios if u['ativo'] and u['cidade']=='Belo Horizonte'];print(resultado);assert resultado==['Ana','Diego']

['Ana', 'Diego']


## Boas práticas

Nunca execute exclusão de documentos em produção como parte de exercícios. Use banco e coleções exclusivos da biblioteca.

## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Esquemas, índices e segurança

MongoDB tem esquema flexível, mas **flexibilidade não elimina contratos de dados**. Considere validação de campos obrigatórios, tipos e índices únicos; use coleções exclusivas de laboratório. O exemplo seguinte valida documentos localmente sem depender do serviço MongoDB.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [6]:
from collections import Counter
ids=[u['_id'] for u in usuarios]
assert len(ids)==len(set(ids)),'IDs duplicados'
assert all(isinstance(u.get('ativo'),bool) for u in usuarios)
print('Documentos válidos:',len(ids))

Documentos válidos: 6


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [7]:
pd.DataFrame(usuarios)[['_id','nome','cidade','ativo']].to_csv(OUT / '07_usuarios_sinteticos_resumo.csv',index=False)
print('Exportado: outputs/07_usuarios_sinteticos_resumo.csv')

Exportado: outputs/07_usuarios_sinteticos_resumo.csv
